# ICACIn Camera-Ready Replication Pipeline — crash-safe V5
This version adds a full pre-run audit, label-free outer/blind inference, atomic result writes, Google Drive persistence after every completed fold, CV checkpoint bundles before final training, and an automatic downloadable results bundle after every completed model.


In [ ]:

# ============================================================
# ICACIn camera-ready replication pipeline V5
# - Source XML -> canonical document-level BIO
# - 5-fold document-level CV on training corpus
# - checkpoint selection on INNER validation token-typed F1 only
# - strict + token metrics from THE SAME outer-fold predictions
# - blind test inference first, gold comparison second
# - one OOF prediction corpus drives every results table
# ============================================================

import os, re, html, json, random, time, zipfile, inspect, csv, sys, platform, hashlib, shutil
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple, Set
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, WeightedRandomSampler

from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, TrainingArguments, Trainer
)
from transformers.trainer_callback import EarlyStoppingCallback

try:
    from google.colab import files
except Exception:
    files = None

# -------------------------
# USER PATHS (Colab defaults)
# -------------------------
TRAIN_XML = "/content/deid_surrogate_train_all_version2.xml"
TEST_BLIND_XML = "/content/deid_surrogate_test_all_version2.xml"
TEST_GOLD_XML = "/content/deid_surrogate_test_all_groundtruth_version2.xml"
OUTPUT_ROOT = "/content/outputs_camera_ready"

DOWNLOAD_RESULTS_JSON = True
DOWNLOAD_RESULTS_BUNDLE = True
DOWNLOAD_FINAL_MODEL_ZIP = False  # huge browser download; final weights are backed up to Drive instead

# Crash protection / persistence
USE_GOOGLE_DRIVE_BACKUP = True
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/ICACIN_CAMERA_READY_BACKUP"
BACKUP_FINAL_MODEL_TO_DRIVE = True
DOWNLOAD_EACH_MODEL_BUNDLE = True
DOWNLOAD_FOLD_BUNDLE_IF_NO_DRIVE = True

_DRIVE_READY = False

SEED = 42

# Six paper categories. Raw PATIENT + DOCTOR are collapsed to NAME.
TARGET_ENTITY_TYPES = ["ID", "NAME", "DATE", "HOSPITAL", "LOCATION", "PHONE"]
RAW_TO_TARGET = {
    "ID": "ID",
    "PATIENT": "NAME",
    "DOCTOR": "NAME",
    "DATE": "DATE",
    "HOSPITAL": "HOSPITAL",
    "LOCATION": "LOCATION",
    "PHONE": "PHONE",
}
EXCLUDED_RAW_TYPES = {"AGE"}

ENTITY_TYPES = list(TARGET_ENTITY_TYPES)
BIO_LABELS = ["O"] + [f"{p}-{e}" for e in ENTITY_TYPES for p in ("B", "I")]
LABEL2ID = {x:i for i,x in enumerate(BIO_LABELS)}
ID2LABEL = {i:x for x,i in LABEL2ID.items()}

# ============================================================
# Reproducibility
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ============================================================
# Helpers
# ============================================================
def _safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def _f1(p, r):
    return (2*p*r/(p+r)) if (p+r) else 0.0

def _make_training_args(**kwargs) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters.keys()); valid.discard("self")
    if "evaluation_strategy" in valid and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    if "eval_strategy" in valid and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**{k:v for k,v in kwargs.items() if k in valid})

def _trainer_tokenizer_kw(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    return {"tokenizer": tokenizer} if "tokenizer" in sig.parameters else {}

def _extract_logits(predictions):
    return predictions[0] if isinstance(predictions, (tuple, list)) else predictions

def _sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def _atomic_json_dump(obj, path, indent=None):
    """Write JSON atomically so a crash cannot leave a half-written result file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def _safe_download(path, label="artifact"):
    """Trigger a Colab browser download without allowing download failure to kill training."""
    if files is None or not path or not os.path.exists(path):
        return False
    try:
        print(f"Downloading {label}: {os.path.basename(path)}")
        files.download(path)
        return True
    except Exception as e:
        print(f"WARNING: browser download failed for {label}: {e}")
        return False

def setup_persistent_backup():
    """Mount Google Drive once. Completed-fold artifacts are copied there immediately."""
    global _DRIVE_READY
    if _DRIVE_READY:
        return True
    if not USE_GOOGLE_DRIVE_BACKUP:
        return False
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_BACKUP_ROOT, exist_ok=True)
        _DRIVE_READY = True
        print("Persistent backup enabled:", DRIVE_BACKUP_ROOT)
    except Exception as e:
        _DRIVE_READY = False
        print("WARNING: Google Drive backup unavailable:", e)
        print("Completed folds will be browser-downloaded instead.")
    return _DRIVE_READY

def _drive_rel(local_path):
    return os.path.relpath(local_path, OUTPUT_ROOT)

def _backup_file_to_drive(local_path):
    if not _DRIVE_READY or not local_path or not os.path.exists(local_path):
        return None
    rel = _drive_rel(local_path)
    dst = os.path.join(DRIVE_BACKUP_ROOT, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(local_path, dst)
    return dst

def _backup_dir_to_drive(local_dir, include_large=True):
    if not _DRIVE_READY or not local_dir or not os.path.isdir(local_dir):
        return None
    rel = _drive_rel(local_dir)
    dst = os.path.join(DRIVE_BACKUP_ROOT, rel)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    ignore = None if include_large else shutil.ignore_patterns("checkpoint-*", "final_model")
    shutil.copytree(local_dir, dst, ignore=ignore)
    return dst

def _restore_compact_model_from_drive(model_dir):
    """Restore completed compact artifacts so a new Colab runtime can resume finished folds/models."""
    if not _DRIVE_READY:
        return False
    rel = _drive_rel(model_dir)
    src = os.path.join(DRIVE_BACKUP_ROOT, rel)
    if not os.path.isdir(src):
        return False
    os.makedirs(model_dir, exist_ok=True)
    for root, dirs, fnames in os.walk(src):
        dirs[:] = [d for d in dirs if not d.startswith("checkpoint-") and d != "final_model"]
        rr = os.path.relpath(root, src)
        dst_root = model_dir if rr == "." else os.path.join(model_dir, rr)
        os.makedirs(dst_root, exist_ok=True)
        for fn in fnames:
            s = os.path.join(root, fn); d = os.path.join(dst_root, fn)
            shutil.copy2(s, d)
    print("Restored completed artifacts from Drive for", os.path.basename(model_dir))
    return True

def _zip_named_files(zip_path, files_to_add, base_dir):
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    tmp = zip_path + ".tmp"
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in files_to_add:
            if p and os.path.exists(p) and os.path.isfile(p):
                z.write(p, arcname=os.path.relpath(p, base_dir))
    os.replace(tmp, zip_path)
    return zip_path

def _persist_completed_fold(model_dir, fold_dir, cfg_name, fold_num):
    """After every successful fold, persist summary + predictions before the next fold starts."""
    summary = os.path.join(fold_dir, "fold_summary.json")
    preds = os.path.join(fold_dir, "heldout_predictions.json")
    if not (os.path.exists(summary) and os.path.exists(preds)):
        raise AssertionError("Fold persistence requested before completed fold files exist")
    integrity = {
        "model": cfg_name, "fold": int(fold_num),
        "summary_sha256": _sha256_file(summary),
        "predictions_sha256": _sha256_file(preds),
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    integrity_path = os.path.join(fold_dir, "fold_integrity.json")
    _atomic_json_dump(integrity, integrity_path, indent=2)
    bundle = os.path.join(model_dir, f"{cfg_name.lower()}_fold_{fold_num}_complete.zip")
    _zip_named_files(bundle, [summary, preds, integrity_path], model_dir)
    if _DRIVE_READY:
        for p in (summary, preds, integrity_path, bundle):
            _backup_file_to_drive(p)
        print(f"[{cfg_name}] fold {fold_num} persisted to Google Drive")
    elif DOWNLOAD_FOLD_BUNDLE_IF_NO_DRIVE:
        _safe_download(bundle, f"{cfg_name} fold {fold_num} backup")
    return bundle

def collect_environment_info():
    try:
        import transformers as _transformers
        transformers_version = _transformers.__version__
    except Exception:
        transformers_version = None
    gpu = None
    if torch.cuda.is_available():
        try:
            gpu = torch.cuda.get_device_name(0)
        except Exception:
            gpu = "CUDA available"
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers_version,
        "numpy": np.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_version": getattr(torch.version, "cuda", None),
        "gpu": gpu,
    }

@dataclass
class DocExample:
    doc_id: str
    words: List[str]
    labels: List[str]

# ============================================================
# XML -> canonical document-level BIO
#
# Why a custom record parser?
# The original train XML contains one malformed nested PHI opening
# in record 236. Parsing the whole file with XML "recover" mode can
# incorrectly absorb later records into that PHI span. We isolate
# RECORD blocks first, then repair nested PHI openings deterministically.
#
# IMPORTANT: tokenization is performed AFTER PHI tags are removed,
# over the complete plain document. This guarantees blind-test and
# gold-test word sequences are identical even when a PHI tag covers
# only part of a whitespace token (e.g. <PHI>6/7</PHI>/1999).
# ============================================================
RECORD_RE = re.compile(
    r'<RECORD\s+ID="([^"]+)">\s*<TEXT>(.*?)</TEXT>\s*</RECORD>',
    re.I | re.S
)
PHI_TAG_RE = re.compile(r'<PHI\s+TYPE="([^"]+)">|</PHI>', re.I)

def _plain_text_and_phi_char_spans(raw_text: str, record_id: str):
    pieces = []
    phi_spans = []  # (start_char, end_char, raw_type)
    repairs = []
    active_type = None
    active_start = None
    cursor = 0
    plain_len = 0

    for m in PHI_TAG_RE.finditer(raw_text):
        seg = html.unescape(raw_text[cursor:m.start()])
        pieces.append(seg)
        plain_len += len(seg)

        opening_type = m.group(1)
        if opening_type is not None:
            opening_type = opening_type.upper()
            if active_type is not None:
                # Nested PHI is invalid in this corpus. In the known malformed
                # record, the nested opening is a mistyped closing tag.
                phi_spans.append((active_start, plain_len, active_type))
                repairs.append({
                    "record_id": record_id,
                    "kind": "nested_open_treated_as_close",
                    "active_type": active_type,
                    "encountered_type": opening_type,
                    "source_char": int(m.start()),
                })
                active_type = None
                active_start = None
                # Deliberately DO NOT open the encountered nested tag.
            else:
                active_type = opening_type
                active_start = plain_len
        else:
            if active_type is None:
                repairs.append({
                    "record_id": record_id,
                    "kind": "orphan_close_ignored",
                    "source_char": int(m.start()),
                })
            else:
                phi_spans.append((active_start, plain_len, active_type))
            active_type = None
            active_start = None

        cursor = m.end()

    seg = html.unescape(raw_text[cursor:])
    pieces.append(seg)
    plain_len += len(seg)

    if active_type is not None:
        phi_spans.append((active_start, plain_len, active_type))
        repairs.append({
            "record_id": record_id,
            "kind": "unclosed_phi_closed_at_record_end",
            "active_type": active_type,
        })

    return "".join(pieces), phi_spans, repairs

def _tokenize_plain(plain_text: str):
    # Whitespace tokenization mirrors the corpus surface form while remaining
    # independent of annotation boundaries.
    matches = list(re.finditer(r'\S+', plain_text))
    return [m.group(0) for m in matches], [(m.start(), m.end()) for m in matches]

def _labels_from_char_spans(token_offsets, phi_spans):
    labels = ["O"] * len(token_offsets)
    collisions = []

    # Linear scan is possible, but corpus size is small enough for this explicit
    # implementation and it is easier to audit.
    ti_start = 0
    for span_i, (s, e, raw_type) in enumerate(phi_spans):
        target = RAW_TO_TARGET.get(raw_type)
        if target is None:
            continue  # AGE/other excluded categories become O

        overlapping = []
        # Advance to first token that could overlap this span.
        while ti_start < len(token_offsets) and token_offsets[ti_start][1] <= s:
            ti_start += 1
        j = ti_start
        while j < len(token_offsets) and token_offsets[j][0] < e:
            ts, te = token_offsets[j]
            if min(te, e) - max(ts, s) > 0:
                overlapping.append(j)
            j += 1

        for k, ti in enumerate(overlapping):
            new_label = ("B-" if k == 0 else "I-") + target
            if labels[ti] != "O" and labels[ti] != new_label:
                collisions.append({
                    "token_index": int(ti),
                    "existing": labels[ti],
                    "new": new_label,
                    "span_index": int(span_i),
                })
            else:
                labels[ti] = new_label

    return labels, collisions

def load_labeled_xml(path: str):
    raw = open(path, "r", encoding="utf-8").read()
    blocks = RECORD_RE.findall(raw)
    if not blocks:
        raise ValueError(f"No RECORD blocks found in {path}")

    docs = []
    repairs = []
    collisions = []

    for record_id, raw_text in blocks:
        plain, phi_spans, rec_repairs = _plain_text_and_phi_char_spans(raw_text, record_id)
        words, offsets = _tokenize_plain(plain)
        labels, rec_collisions = _labels_from_char_spans(offsets, phi_spans)

        if len(words) != len(labels):
            raise AssertionError(f"word/label mismatch in record {record_id}")

        docs.append(DocExample(str(record_id), words, labels))
        repairs.extend(rec_repairs)
        for c in rec_collisions:
            c["record_id"] = str(record_id)
            collisions.append(c)

    ids = [d.doc_id for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate RECORD IDs in labeled XML")

    return docs, {"path": path, "records": len(docs), "repairs": repairs, "collisions": collisions}

def load_blind_xml(path: str):
    raw = open(path, "r", encoding="utf-8").read()
    blocks = RECORD_RE.findall(raw)
    if not blocks:
        raise ValueError(f"No RECORD blocks found in {path}")

    docs = []
    for record_id, raw_text in blocks:
        # Blind source has no PHI tags. Dummy O labels are only used so the
        # token-classification dataset can run inference; they are NEVER scored.
        plain = html.unescape(raw_text)
        words, _ = _tokenize_plain(plain)
        docs.append(DocExample(str(record_id), words, ["O"] * len(words)))

    ids = [d.doc_id for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate RECORD IDs in blind XML")
    return docs

# ============================================================
# BIO audit / repair
# ============================================================
def repair_bio_sequence(tags: List[str]) -> Tuple[List[str], int]:
    out = []
    changed = 0
    prev_type = None
    prev_inside = False

    for t in tags:
        if t == "O" or "-" not in t:
            out.append("O" if t == "O" else t)
            prev_type = None
            prev_inside = False
            continue

        pref, et = t.split("-", 1)
        if pref == "I" and (not prev_inside or prev_type != et):
            t = f"B-{et}"
            changed += 1
            pref = "B"

        out.append(t)
        prev_type = et
        prev_inside = pref in ("B", "I")

    return out, changed

def audit_and_repair_docs(docs: List[DocExample]):
    changed = 0
    total = 0
    out = []
    for d in docs:
        labs, ch = repair_bio_sequence(d.labels)
        changed += ch
        total += len(labs)
        out.append(DocExample(d.doc_id, d.words, labs))
    return out, {"bio_repairs": int(changed), "word_tokens": int(total)}

# ============================================================
# Dataset audit
# ============================================================
def extract_entities_bio(tags: List[str]) -> List[Tuple[int, int, str]]:
    tags_rep, _ = repair_bio_sequence(tags)
    spans = []
    cur_type = None
    start = None

    def close(i):
        nonlocal cur_type, start
        if cur_type is not None and start is not None:
            spans.append((start, i, cur_type))
        cur_type, start = None, None

    for i, t in enumerate(tags_rep):
        if t == "O":
            close(i)
            continue
        if "-" not in t:
            close(i)
            continue
        pref, et = t.split("-", 1)
        if pref == "B":
            close(i)
            cur_type, start = et, i
        elif pref == "I" and cur_type == et and start is not None:
            continue
        elif pref == "I":
            close(i)
            cur_type, start = et, i
        else:
            close(i)
    close(len(tags_rep))
    return spans

def entity_counts_from_docs(docs: List[DocExample]):
    c = Counter()
    for d in docs:
        c.update(t for _,_,t in extract_entities_bio(d.labels))
    return {e:int(c.get(e, 0)) for e in ENTITY_TYPES}

def prepare_corpora(train_xml=TRAIN_XML, blind_xml=TEST_BLIND_XML, gold_xml=TEST_GOLD_XML):
    train_docs, train_parse = load_labeled_xml(train_xml)
    blind_docs = load_blind_xml(blind_xml)

    train_docs, train_bio = audit_and_repair_docs(train_docs)

    train_ids = {d.doc_id for d in train_docs}
    test_ids = {d.doc_id for d in blind_docs}
    overlap = train_ids & test_ids
    if overlap:
        raise AssertionError(f"Train/test RECORD ID overlap: {len(overlap)}")

    audit = {
        "train_records": len(train_docs),
        "blind_test_records": len(blind_docs),
        "train_test_id_overlap": 0,
        "train_word_tokens": sum(len(d.words) for d in train_docs),
        "train_entity_counts": entity_counts_from_docs(train_docs),
        "xml_repairs": train_parse["repairs"],
        "token_label_collisions": train_parse["collisions"],
        **train_bio,
    }

    print("DATASET AUDIT")
    print(json.dumps(audit, indent=2))
    print("\nGold test is intentionally NOT loaded here. It is opened only after blind predictions are produced.")
    return train_docs, blind_docs, audit

# ============================================================
# Tokenizer + sliding windows
#
# This retains the effective layout of the older implementation:
#   256 original word tokens/window, 128-word step,
#   tokenizer capped at 512 subword pieces.
# The paper should describe these units accurately.
# ============================================================
def build_tokenizer(model_id: str):
    kwargs = {}
    if "roberta" in model_id.lower():
        kwargs["add_prefix_space"] = True
    return AutoTokenizer.from_pretrained(model_id, use_fast=True, **kwargs)

class SlidingWindowDataset(Dataset):
    """Document windows with tokenizer-aware subword budgeting.

    Design goal: preserve the older experiment's ~256-word / 128-word-step
    layout, while NEVER silently truncating original words at the 512-subword
    model boundary. Candidate windows are shortened only when necessary so
    that every selected original word is fully represented by the tokenizer.
    """
    def __init__(
        self, docs: List[DocExample], tokenizer,
        window_words: int = 256, step_words: int = 128, max_subwords: int = 512,
        compute_sample_weights: bool = False,
        window_boosts: Optional[Dict[str,float]] = None
    ):
        self.docs = docs
        self.tok = tokenizer
        self.window_words = int(window_words)
        self.step_words = int(step_words)
        self.max_subwords = int(max_subwords)
        self.window_boosts = window_boosts or {}
        self.windows = []
        self._word_ids_cache = []

        for di, d in enumerate(docs):
            n = len(d.words)
            s = 0
            while s < n:
                target_e = min(n, s + self.window_words)
                e = self._largest_fitting_end(d.words, s, target_e)
                if e <= s:
                    raise AssertionError(
                        f"Tokenizer could not fit even one word: doc={d.doc_id}, start={s}"
                    )
                self.windows.append({"doc_i":di, "start":s, "end":e})

                if e >= n:
                    break

                # Retain the intended 128-word step when it cannot create a gap.
                # If subword expansion forced an unusually short window, advance
                # only to its end so every original word still receives coverage.
                proposed = s + self.step_words
                s_next = proposed if proposed <= e else e
                if s_next <= s:
                    raise AssertionError(f"Window builder made no progress: doc={d.doc_id}, start={s}")
                s = s_next

        self._word_ids_cache = [None] * len(self.windows)

        self.sample_weights = None
        if compute_sample_weights:
            self.sample_weights = []
            for w in self.windows:
                d = self.docs[w["doc_i"]]
                seg = d.labels[w["start"]:w["end"]]
                weight = 1.0
                # Preserve the older implementation's training policy so the
                # replication does not silently change minority handling.
                for t in seg:
                    if t == "O":
                        continue
                    et = t.split("-",1)[1]
                    weight *= float(self.window_boosts.get(et, 1.0))
                self.sample_weights.append(min(10.0, max(0.1, weight)))

    def _encoded_length(self, words: List[str]) -> int:
        enc = self.tok(
            words,
            is_split_into_words=True,
            truncation=False,
            add_special_tokens=True,
            return_attention_mask=False,
        )
        return len(enc["input_ids"])

    def _largest_fitting_end(self, words: List[str], start: int, target_end: int) -> int:
        """Largest end <= target_end whose complete encoding fits max_subwords."""
        if start >= target_end:
            return start

        # Fast path: normal 256-word candidate fits.
        if self._encoded_length(words[start:target_end]) <= self.max_subwords:
            return target_end

        # Binary-search the longest complete-word prefix that fits.
        lo, hi = start + 1, target_end
        best = start
        while lo <= hi:
            mid = (lo + hi) // 2
            L = self._encoded_length(words[start:mid])
            if L <= self.max_subwords:
                best = mid
                lo = mid + 1
            else:
                hi = mid - 1
        return best

    def __len__(self):
        return len(self.windows)

    def _encode_window(self, idx):
        w = self.windows[idx]
        d = self.docs[w["doc_i"]]
        words = d.words[w["start"]:w["end"]]
        tags  = d.labels[w["start"]:w["end"]]

        # No truncation here: the constructor has already guaranteed that this
        # complete-word window fits the model subword budget.
        enc = self.tok(
            words,
            is_split_into_words=True,
            truncation=False,
            max_length=self.max_subwords,
            return_attention_mask=True,
        )
        if len(enc["input_ids"]) > self.max_subwords:
            raise AssertionError(
                f"Window exceeded subword budget after planning: doc={d.doc_id}, "
                f"start={w['start']}, end={w['end']}, subwords={len(enc['input_ids'])}"
            )

        word_ids = enc.word_ids()
        represented = {int(x) for x in word_ids if x is not None}
        expected = set(range(len(words)))
        if represented != expected:
            missing = sorted(expected - represented)
            raise AssertionError(
                f"Tokenizer failed to represent original words: doc={d.doc_id}, "
                f"window={w['start']}:{w['end']}, missing_local={missing[:10]}"
            )
        self._word_ids_cache[idx] = word_ids

        labels = []
        seen_words = set()
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid not in seen_words:
                labels.append(LABEL2ID[tags[wid]])
                seen_words.add(wid)
            else:
                labels.append(-100)

        return enc, labels

    def get_padded_word_ids(self, idx, target_len):
        if self._word_ids_cache[idx] is None:
            self._encode_window(idx)
        wid = self._word_ids_cache[idx]
        if len(wid) < target_len:
            return wid + [None] * (target_len - len(wid))
        return wid[:target_len]

    def coverage_audit(self, verify_tokenizer: bool = True):
        """Fail before training if any original word is not safely represented."""
        doc_counts = [np.zeros(len(d.words), dtype=np.int32) for d in self.docs]
        max_seq_len = 0
        shortened = 0

        for i, w in enumerate(self.windows):
            d = self.docs[w["doc_i"]]
            planned_words = w["end"] - w["start"]
            if planned_words < min(self.window_words, len(d.words) - w["start"]):
                shortened += 1

            if verify_tokenizer:
                enc, _ = self._encode_window(i)
                max_seq_len = max(max_seq_len, len(enc["input_ids"]))
                word_ids = enc.word_ids()
                represented = {int(x) for x in word_ids if x is not None}
                for wid in represented:
                    abs_w = w["start"] + wid
                    if 0 <= abs_w < len(doc_counts[w["doc_i"]]):
                        doc_counts[w["doc_i"]][abs_w] += 1
            else:
                doc_counts[w["doc_i"]][w["start"]:w["end"]] += 1

        problems = []
        for di, cnt in enumerate(doc_counts):
            missing = np.where(cnt == 0)[0]
            if len(missing):
                problems.append({
                    "record_id": self.docs[di].doc_id,
                    "count": int(len(missing)),
                    "first": missing[:10].tolist(),
                })
        if problems:
            raise AssertionError(f"Tokenizer/window preflight found uncovered words: {problems[:3]}")

        return {
            "documents": int(len(self.docs)),
            "windows": int(len(self.windows)),
            "shortened_windows_due_to_subword_budget": int(shortened),
            "max_encoded_subwords": int(max_seq_len),
            "uncovered_words": 0,
        }

    def __getitem__(self, idx):
        enc, labels = self._encode_window(idx)
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ============================================================
# Optional class-weighted loss (preserved from older run)
# ============================================================
def compute_class_weights_from_docs(
    docs: List[DocExample], o_mult=0.7, clip_min=0.2, clip_max=5.0, **entity_mults
):
    counts = Counter()
    for d in docs:
        counts.update(d.labels)
    total = sum(counts.values())
    weights = []
    for lab in BIO_LABELS:
        c = counts.get(lab, 0)
        w = total / max(1, c)
        weights.append(w)
    w = np.asarray(weights, dtype=np.float64)
    w = w / max(1e-12, np.median(w[w > 0]))
    w = np.clip(w, clip_min, clip_max)
    w[LABEL2ID["O"]] *= float(o_mult)

    for et, mult in entity_mults.items():
        for pref in ("B","I"):
            lab = f"{pref}-{et}"
            if lab in LABEL2ID:
                w[LABEL2ID[lab]] *= float(mult)
    return torch.tensor(w, dtype=torch.float32)

class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is None:
            loss = outputs.loss
        else:
            cw = self.class_weights.to(logits.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=cw, ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

class PredictionOnlyDataset(Dataset):
    """Inference view that strips gold labels before outer/blind model calls."""
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        item = self.base[idx]
        return {k:v for k,v in item.items() if k != "labels"}

# ============================================================
# Merge window logits -> one document prediction
# ============================================================
def merge_logits_to_docs(dataset: SlidingWindowDataset, logits, label_ids=None):
    """Merge overlapping-window logits back to original words.

    IMPORTANT: coverage/first-subword selection is derived ONLY from tokenizer
    word_ids, never from returned gold label_ids. Gold labels are not needed to
    decide which model position represents an original word.
    """
    doc_sums = []
    doc_cnts = []
    for d in dataset.docs:
        n = len(d.words)
        doc_sums.append(np.zeros((n, len(BIO_LABELS)), dtype=np.float64))
        doc_cnts.append(np.zeros(n, dtype=np.float64))

    if len(logits) != len(dataset):
        raise AssertionError(f"Prediction/window count mismatch: {len(logits)} vs {len(dataset)}")

    for i in range(len(dataset)):
        meta = dataset.windows[i]
        di = meta["doc_i"]
        start = meta["start"]
        seq_len = logits[i].shape[0]
        word_ids = dataset.get_padded_word_ids(i, seq_len)

        # Use exactly the first model subtoken for each original word.
        # This mirrors first-subword supervision and is independent of gold labels.
        seen_local = set()
        for pos, wid in enumerate(word_ids):
            if wid is None or wid in seen_local:
                continue
            seen_local.add(wid)
            abs_w = start + int(wid)
            if 0 <= abs_w < len(doc_cnts[di]):
                doc_sums[di][abs_w] += logits[i][pos]
                doc_cnts[di][abs_w] += 1.0

    doc_order, yt_docs, yp_docs = [], [], []
    for di, d in enumerate(dataset.docs):
        if len(d.words) == 0:
            continue
        uncovered = np.where(doc_cnts[di] == 0)[0]
        if len(uncovered):
            raise AssertionError(
                f"Uncovered words after tokenizer-only window merge: doc={d.doc_id}, "
                f"count={len(uncovered)}, first={uncovered[:10].tolist()}"
            )
        avg = doc_sums[di] / doc_cnts[di][:,None]
        pred_ids = avg.argmax(axis=-1)
        yp = [ID2LABEL[int(x)] for x in pred_ids]
        doc_order.append(d.doc_id)
        yt_docs.append(d.labels)
        yp_docs.append(yp)
    return doc_order, yt_docs, yp_docs

# ============================================================
# Metrics
# ============================================================
def _to_type(tag):
    return "O" if tag == "O" or "-" not in tag else tag.split("-",1)[1]

def token_typed_breakdown(yt_docs, yp_docs):
    tp = fp = fn = 0
    per = {e:{"tp":0,"fp":0,"fn":0,"gold":0,"pred":0} for e in ENTITY_TYPES}

    for yt, yp in zip(yt_docs, yp_docs):
        if len(yt) != len(yp):
            raise AssertionError("gold/pred token length mismatch")
        for g,p in zip(yt,yp):
            gt, pt = _to_type(g), _to_type(p)
            if gt != "O": per[gt]["gold"] += 1
            if pt != "O": per[pt]["pred"] += 1
            if gt == pt and gt != "O":
                tp += 1; per[gt]["tp"] += 1
            else:
                if pt != "O": fp += 1; per[pt]["fp"] += 1
                if gt != "O": fn += 1; per[gt]["fn"] += 1

    return _finish_breakdown(tp, fp, fn, per)

def entity_exact_breakdown(yt_docs, yp_docs):
    tp = fp = fn = 0
    per = {e:{"tp":0,"fp":0,"fn":0,"gold":0,"pred":0} for e in ENTITY_TYPES}

    for yt, yp in zip(yt_docs, yp_docs):
        gs = set(extract_entities_bio(yt))
        ps = set(extract_entities_bio(yp))
        inter = gs & ps
        tp += len(inter); fp += len(ps-gs); fn += len(gs-ps)

        for _,_,t in gs:
            if t in per: per[t]["gold"] += 1
        for _,_,t in ps:
            if t in per: per[t]["pred"] += 1
        for _,_,t in inter:
            if t in per: per[t]["tp"] += 1
        for _,_,t in ps-gs:
            if t in per: per[t]["fp"] += 1
        for _,_,t in gs-ps:
            if t in per: per[t]["fn"] += 1

    return _finish_breakdown(tp, fp, fn, per)

def _finish_breakdown(tp, fp, fn, per):
    P = _safe_div(tp, tp+fp)
    R = _safe_div(tp, tp+fn)
    out = {}
    macro = []
    for e,c in per.items():
        p = _safe_div(c["tp"], c["tp"]+c["fp"])
        r = _safe_div(c["tp"], c["tp"]+c["fn"])
        f = _f1(p,r)
        macro.append(f)
        gold, pred = c["gold"], c["pred"]
        out[e] = {
            **{k:int(v) for k,v in c.items()},
            "precision":float(p), "recall":float(r), "f1":float(f),
            "fp_share":float(_safe_div(c["fp"], c["tp"]+c["fp"])),
            "fn_share":float(_safe_div(c["fn"], c["tp"]+c["fn"])),
            "bias":int(pred-gold),
            "bias_pct":float(_safe_div(pred-gold, gold)),
        }
    return {
        "overall":{
            "precision":float(P), "recall":float(R), "f1":float(_f1(P,R)),
            "macro_f1":float(np.mean(macro)), "tp":int(tp), "fp":int(fp), "fn":int(fn)
        },
        "per_entity":out
    }

RISK_MULTIPLIERS = {
    "ID":3.0, "NAME":3.0, "PHONE":2.0,
    "LOCATION":2.0, "HOSPITAL":1.5, "DATE":1.0
}

def risk_weighted_f1(strict_breakdown):
    num = den = 0.0
    for e in ENTITY_TYPES:
        row = strict_breakdown["per_entity"][e]
        gold = float(row["gold"])
        w = float(RISK_MULTIPLIERS[e])
        num += gold * w * float(row["f1"])
        den += gold * w
    return _safe_div(num, den)

# ============================================================
# CV split
# Keep the seeded document-level folds from the older experiment.
# Do not relabel them "entity-stratified" in the paper.
# ============================================================
def build_folds_doclevel(n_docs, n_folds, seed):
    idx = list(range(n_docs))
    random.Random(seed).shuffle(idx)
    folds = [[] for _ in range(n_folds)]
    for i,v in enumerate(idx):
        folds[i % n_folds].append(v)
    return folds

def split_inner_val_doclevel(train_fold_docs, val_frac, seed):
    idx = list(range(len(train_fold_docs)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(round(val_frac * len(idx))))
    val_idx = set(idx[:n_val])
    tr = [d for i,d in enumerate(train_fold_docs) if i not in val_idx]
    va = [d for i,d in enumerate(train_fold_docs) if i in val_idx]
    return tr, va

# ============================================================
# Model config
# Hyperparameters below retain the older implementation so this is
# a replication/correction run, not a new post-hoc experiment.
# ============================================================
@dataclass
class ModelCfg:
    name: str
    model_id: str
    window_words: int = 256
    step_words: int = 128
    max_subwords: int = 512

    batch_size: int = 4
    grad_accum: int = 2
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    lr_scheduler_type: str = "linear"
    max_grad_norm: float = 1.0
    fp16: bool = True

    epochs_cv: int = 8
    epochs_final: int = 10
    n_folds: int = 5
    inner_val_frac: float = 0.10

    window_boosts: Optional[Dict[str,float]] = None
    use_class_weights: bool = True
    class_weight_multipliers: Optional[Dict[str,float]] = None

    early_stop_patience: int = 3
    early_stop_threshold: float = 0.0

def cfg_defaults(cfg):
    if cfg.window_boosts is None:
        cfg.window_boosts = {"LOCATION":1.3, "PHONE":1.0}
    if cfg.class_weight_multipliers is None:
        cfg.class_weight_multipliers = {"LOCATION":1.5, "PHONE":1.1}
    return cfg

# ============================================================
# Train one outer fold
#
# CRITICAL CORRECTION:
# - checkpoint selection ALWAYS uses inner-validation TOKEN-TYPED F1
# - outer fold predicted ONCE
# - strict and token metrics are calculated from the SAME predictions
# ============================================================
def predict_without_metrics(trainer, dataset, metric_key_prefix="predict"):
    """Run inference without calling the Trainer's validation compute_metrics closure.

    The fold trainer's compute_metrics function is intentionally bound to the INNER
    validation dataset for checkpoint selection. Hugging Face Trainer.predict() would
    otherwise call that same function on OUTER-held-out logits, causing a
    window-count mismatch. Prediction scoring is performed explicitly after merging.
    """
    original_compute_metrics = trainer.compute_metrics
    trainer.compute_metrics = None
    try:
        out = trainer.predict(dataset, metric_key_prefix=metric_key_prefix)
        logits = _extract_logits(out.predictions)
        if not np.isfinite(logits).all():
            raise AssertionError(f"Non-finite logits detected during {metric_key_prefix} inference")
        return out
    finally:
        trainer.compute_metrics = original_compute_metrics

def train_one_fold(cfg, tokenizer, train_fold_docs, heldout_docs, out_dir, seed):
    cfg = cfg_defaults(cfg)
    os.makedirs(out_dir, exist_ok=True)
    tr_in, va_in = split_inner_val_doclevel(
        train_fold_docs, cfg.inner_val_frac, seed + 1337
    )

    train_ds = SlidingWindowDataset(
        tr_in, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords,
        compute_sample_weights=True, window_boosts=cfg.window_boosts
    )
    val_ds = SlidingWindowDataset(
        va_in, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords
    )
    held_ds = SlidingWindowDataset(
        heldout_docs, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords
    )

    weights = torch.tensor(train_ds.sample_weights, dtype=torch.double)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    config = AutoConfig.from_pretrained(
        cfg.model_id, num_labels=len(BIO_LABELS),
        id2label=ID2LABEL, label2id=LABEL2ID
    )
    model = AutoModelForTokenClassification.from_pretrained(cfg.model_id, config=config)
    collator = DataCollatorForTokenClassification(tokenizer)

    class_weights = None
    if cfg.use_class_weights:
        class_weights = compute_class_weights_from_docs(
            tr_in, o_mult=0.7, clip_min=0.2, clip_max=5.0,
            **dict(cfg.class_weight_multipliers or {})
        )

    args = _make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs_cv,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,
        max_grad_norm=cfg.max_grad_norm,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=seed,
        report_to="none",
        save_total_limit=1,
        logging_strategy="epoch",
        fp16=(cfg.fp16 and torch.cuda.is_available()),
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        logits = _extract_logits(logits)
        _, yt, yp = merge_logits_to_docs(val_ds, logits, labels)
        token = token_typed_breakdown(yt, yp)
        # f1 used for checkpoint selection is token-typed only.
        return {
            "precision":token["overall"]["precision"],
            "recall":token["overall"]["recall"],
            "f1":token["overall"]["f1"],
            "macro_f1":token["overall"]["macro_f1"],
        }

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=cfg.early_stop_patience,
            early_stopping_threshold=cfg.early_stop_threshold
        )],
        class_weights=class_weights,
        **_trainer_tokenizer_kw(tokenizer)
    )

    def _get_train_dataloader_override():
        return torch.utils.data.DataLoader(
            train_ds, batch_size=cfg.batch_size,
            sampler=sampler, collate_fn=collator
        )
    trainer.get_train_dataloader = _get_train_dataloader_override

    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0

    # IMPORTANT: the trainer's compute_metrics closure is bound to val_ds for
    # checkpoint selection. Disable it during outer-fold prediction; otherwise
    # Trainer.predict(held_ds) would incorrectly score heldout logits as if they
    # belonged to val_ds. We score the outer fold explicitly below.
    pred = predict_without_metrics(trainer, PredictionOnlyDataset(held_ds), metric_key_prefix="outer")
    logits = _extract_logits(pred.predictions)
    if len(logits) != len(held_ds):
        raise AssertionError(
            f"Outer prediction/window count mismatch before merge: "
            f"logits={len(logits)}, windows={len(held_ds)}, heldout_docs={len(heldout_docs)}"
        )
    doc_order, yt, yp = merge_logits_to_docs(held_ds, logits, None)

    strict = entity_exact_breakdown(yt, yp)
    token = token_typed_breakdown(yt, yp)

    predictions = []
    words_map = {d.doc_id:d.words for d in heldout_docs}
    for doc_id, gold, guess in zip(doc_order, yt, yp):
        predictions.append({
            "record_id":doc_id,
            "tokens":words_map[doc_id],
            "gold":gold,
            "pred":guess,
        })

    fold_summary = {
        "train_fold_docs":len(train_fold_docs),
        "train_inner_docs":len(tr_in),
        "val_inner_docs":len(va_in),
        "heldout_docs":len(heldout_docs),
        "train_inner_record_ids":[d.doc_id for d in tr_in],
        "val_inner_record_ids":[d.doc_id for d in va_in],
        "heldout_record_ids":[d.doc_id for d in heldout_docs],
        "best_checkpoint":getattr(trainer.state, "best_model_checkpoint", None),
        "best_metric":None if getattr(trainer.state, "best_metric", None) is None else float(trainer.state.best_metric),
        "global_step":int(getattr(trainer.state, "global_step", 0)),
        "train_seconds":float(train_seconds),
        "strict":strict,
        "token":token,
        "trainer_log_history":getattr(trainer.state, "log_history", []),
    }

    # Save each fold immediately so an interrupted multi-fold run still leaves
    # reusable held-out predictions, split membership, and training history.
    fold_pred_file = os.path.join(out_dir, "heldout_predictions.json")
    _atomic_json_dump(predictions, fold_pred_file)
    fold_summary_file = os.path.join(out_dir, "fold_summary.json")
    _atomic_json_dump(fold_summary, fold_summary_file, indent=2)

    return {
        **fold_summary,
        "fold_summary_file":fold_summary_file,
        "heldout_predictions_file":fold_pred_file,
        "predictions":predictions,
    }

# ============================================================
# Full-train model -> BLIND test predictions -> only then load gold
# ============================================================
def train_final_and_score_blind(cfg, tokenizer, train_docs, blind_docs, gold_xml, out_dir, seed):
    cfg = cfg_defaults(cfg)
    os.makedirs(out_dir, exist_ok=True)

    train_ds = SlidingWindowDataset(
        train_docs, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords,
        compute_sample_weights=True, window_boosts=cfg.window_boosts
    )
    blind_ds = SlidingWindowDataset(
        blind_docs, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords
    )

    weights = torch.tensor(train_ds.sample_weights, dtype=torch.double)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    class_weights = None
    if cfg.use_class_weights:
        class_weights = compute_class_weights_from_docs(
            train_docs, o_mult=0.7, clip_min=0.2, clip_max=5.0,
            **dict(cfg.class_weight_multipliers or {})
        )

    config = AutoConfig.from_pretrained(
        cfg.model_id, num_labels=len(BIO_LABELS),
        id2label=ID2LABEL, label2id=LABEL2ID
    )
    model = AutoModelForTokenClassification.from_pretrained(cfg.model_id, config=config)
    collator = DataCollatorForTokenClassification(tokenizer)

    args = _make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.epochs_final,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        lr_scheduler_type=cfg.lr_scheduler_type,
        max_grad_norm=cfg.max_grad_norm,
        eval_strategy="no",
        save_strategy="epoch",
        load_best_model_at_end=False,
        seed=seed,
        report_to="none",
        save_total_limit=1,
        logging_strategy="epoch",
        fp16=(cfg.fp16 and torch.cuda.is_available()),
    )

    trainer = CustomTrainer(
        model=model, args=args, train_dataset=train_ds,
        data_collator=collator, class_weights=class_weights,
        **_trainer_tokenizer_kw(tokenizer)
    )

    def _get_train_dataloader_override():
        return torch.utils.data.DataLoader(
            train_ds, batch_size=cfg.batch_size,
            sampler=sampler, collate_fn=collator
        )
    trainer.get_train_dataloader = _get_train_dataloader_override

    final_t0 = time.time()
    trainer.train()
    final_train_seconds = time.time() - final_t0

    final_training_summary_file = os.path.join(out_dir, "final_training_summary.json")
    _atomic_json_dump({
        "train_docs": len(train_docs),
        "train_windows": len(train_ds),
        "global_step": int(getattr(trainer.state, "global_step", 0)),
        "train_seconds": float(final_train_seconds),
        "trainer_log_history": getattr(trainer.state, "log_history", []),
        "epochs_requested": int(cfg.epochs_final),
        "checkpoint_selection": "none; fixed full-data training epochs for blind-test model",
    }, final_training_summary_file, indent=2)

    # 1) BLIND PREDICTION FIRST
    blind_pred = predict_without_metrics(trainer, PredictionOnlyDataset(blind_ds), metric_key_prefix="blind")
    logits = _extract_logits(blind_pred.predictions)
    if len(logits) != len(blind_ds):
        raise AssertionError(
            f"Blind prediction/window count mismatch before merge: "
            f"logits={len(logits)}, windows={len(blind_ds)}, blind_docs={len(blind_docs)}"
        )
    doc_order, _, yp = merge_logits_to_docs(blind_ds, logits, None)

    pred_by_id = {doc_id:pred for doc_id,pred in zip(doc_order,yp)}
    blind_words = {d.doc_id:d.words for d in blind_docs}

    blind_prediction_file = os.path.join(out_dir, "blind_predictions.json")
    _atomic_json_dump([
        {"record_id":rid, "tokens":blind_words[rid], "pred":pred_by_id[rid]}
        for rid in doc_order
    ], blind_prediction_file)

    # 2) GOLD IS OPENED ONLY AFTER PREDICTIONS ARE FROZEN
    gold_docs, gold_parse = load_labeled_xml(gold_xml)
    gold_docs, gold_bio = audit_and_repair_docs(gold_docs)

    gold_by_id = {d.doc_id:d for d in gold_docs}
    if set(gold_by_id) != set(pred_by_id):
        raise AssertionError("Blind/gold RECORD ID sets do not match")

    yt, aligned_yp = [], []
    for rid in doc_order:
        g = gold_by_id[rid]
        if g.words != blind_words[rid]:
            raise AssertionError(
                f"Blind/gold tokenization mismatch in record {rid}. "
                "Canonical tokenization must be annotation-boundary independent."
            )
        yt.append(g.labels)
        aligned_yp.append(pred_by_id[rid])

    strict = entity_exact_breakdown(yt, aligned_yp)
    token = token_typed_breakdown(yt, aligned_yp)

    final_scored_file = os.path.join(out_dir, "final_test_predictions_with_gold.json")
    _atomic_json_dump([
        {
            "record_id":rid,
            "tokens":blind_words[rid],
            "gold":gold_by_id[rid].labels,
            "pred":pred_by_id[rid],
        }
        for rid in doc_order
    ], final_scored_file)

    final_model_dir = os.path.join(out_dir, "final_model")
    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    return {
        "blind_prediction_file":blind_prediction_file,
        "final_test_predictions_with_gold_file":final_scored_file,
        "final_model_dir":final_model_dir,
        "final_training_summary_file":final_training_summary_file,
        "gold_records":len(gold_docs),
        "gold_word_tokens":sum(len(d.words) for d in gold_docs),
        "gold_entity_counts":entity_counts_from_docs(gold_docs),
        "gold_xml_repairs":gold_parse["repairs"],
        "gold_bio_audit":gold_bio,
        "strict":strict,
        "token":token,
    }

# ============================================================
# One model: 5-fold OOF + final blind test
# ============================================================
def run_model(cfg, seed=SEED, train_xml=TRAIN_XML, blind_xml=TEST_BLIND_XML, gold_xml=TEST_GOLD_XML):
    set_seed(seed)
    cfg = cfg_defaults(cfg)
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    setup_persistent_backup()

    train_docs, blind_docs, dataset_audit = prepare_corpora(train_xml, blind_xml, gold_xml)
    tokenizer = build_tokenizer(cfg.model_id)

    # Model-specific tokenizer/window preflight BEFORE any expensive training.
    preflight_train_ds = SlidingWindowDataset(
        train_docs, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords
    )
    preflight_blind_ds = SlidingWindowDataset(
        blind_docs, tokenizer, cfg.window_words, cfg.step_words, cfg.max_subwords
    )
    tokenizer_window_audit = {
        "train": preflight_train_ds.coverage_audit(verify_tokenizer=True),
        "blind_test": preflight_blind_ds.coverage_audit(verify_tokenizer=True),
    }
    print("TOKENIZER/WINDOW PREFLIGHT")
    print(json.dumps(tokenizer_window_audit, indent=2))

    model_dir = os.path.join(OUTPUT_ROOT, cfg.name.lower())
    os.makedirs(model_dir, exist_ok=True)
    _restore_compact_model_from_drive(model_dir)

    # A fully completed model result can be reused after a runtime restart.
    completed_result_file = os.path.join(model_dir, f"{cfg.name.lower()}_camera_ready_results.json")
    if os.path.exists(completed_result_file):
        try:
            old_result = json.load(open(completed_result_file, encoding="utf-8"))
            if (
                old_result.get("schema_version") == "icacin_camera_ready_replication_v5"
                and old_result.get("seed") == seed
                and old_result.get("model_cfg") == asdict(cfg)
                and old_result.get("source_files", {}).get("train_xml", {}).get("sha256") == _sha256_file(train_xml)
                and old_result.get("source_files", {}).get("blind_test_xml", {}).get("sha256") == _sha256_file(blind_xml)
                and old_result.get("source_files", {}).get("gold_test_xml", {}).get("sha256") == _sha256_file(gold_xml)
            ):
                print(f"[{cfg.name}] complete V5 result already exists; reusing it.")
                existing_bundle = os.path.join(model_dir, f"{cfg.name.lower()}_complete_results.zip")
                if DOWNLOAD_RESULTS_JSON:
                    _safe_download(completed_result_file, f"{cfg.name} summary JSON")
                if DOWNLOAD_EACH_MODEL_BUNDLE and os.path.exists(existing_bundle):
                    _safe_download(existing_bundle, f"{cfg.name} complete results bundle")
                return completed_result_file
        except Exception as e:
            print(f"[{cfg.name}] existing result could not be validated; continuing from completed folds: {e}")

    folds = build_folds_doclevel(len(train_docs), cfg.n_folds, seed)
    fold_results = []
    oof = []

    for fi, held_idx in enumerate(folds, start=1):
        held_set = set(held_idx)
        tr = [d for i,d in enumerate(train_docs) if i not in held_set]
        ho = [d for i,d in enumerate(train_docs) if i in held_set]
        print(f"\n[{cfg.name}] fold {fi}/{cfg.n_folds}: train={len(tr)}, heldout={len(ho)}")
        fold_dir = os.path.join(model_dir, f"fold_{fi}")
        summary_path = os.path.join(fold_dir, "fold_summary.json")
        pred_path = os.path.join(fold_dir, "heldout_predictions.json")

        # Resume only fully completed folds whose held-out IDs match this exact split.
        if os.path.exists(summary_path) and os.path.exists(pred_path):
            saved_summary = json.load(open(summary_path, encoding="utf-8"))
            saved_preds = json.load(open(pred_path, encoding="utf-8"))
            expected_ids = [d.doc_id for d in ho]
            saved_ids = [x["record_id"] for x in saved_preds]
            if saved_summary.get("heldout_record_ids") == expected_ids and saved_ids == expected_ids:
                print(f"[{cfg.name}] fold {fi}: reusing completed saved fold")
                r = saved_summary
                fold_predictions = saved_preds
            else:
                raise AssertionError(
                    f"Existing fold_{fi} files do not match current split; remove that fold directory before rerunning."
                )
        else:
            # A prior crash may have left checkpoint-* files without a completed
            # held-out result. Never mix those partial artifacts into a fresh fold.
            if os.path.isdir(fold_dir):
                print(f"[{cfg.name}] fold {fi}: removing incomplete prior fold directory")
                shutil.rmtree(fold_dir)
            r = train_one_fold(
                cfg, tokenizer, tr, ho, fold_dir, seed + fi
            )
            fold_predictions = r.pop("predictions")

        r["fold"] = fi
        for item in fold_predictions:
            item["fold"] = fi
        oof.extend(fold_predictions)
        fold_results.append(r)
        print(
            f"strict={r['strict']['overall']['f1']:.6f} | "
            f"token={r['token']['overall']['f1']:.6f}"
        )
        # Crash barrier: do not begin the next fold until this completed fold
        # has been persisted externally (Drive, or browser download fallback).
        _persist_completed_fold(model_dir, fold_dir, cfg.name, fi)

    # Assert each training record appears exactly once OOF.
    oof_ids = [x["record_id"] for x in oof]
    if len(oof_ids) != len(train_docs) or len(set(oof_ids)) != len(train_docs):
        raise AssertionError("OOF predictions must contain every training record exactly once")

    # Restore source train order before aggregate scoring.
    oof_map = {x["record_id"]:x for x in oof}
    ordered_oof = [oof_map[d.doc_id] for d in train_docs]
    oof_gold = [x["gold"] for x in ordered_oof]
    oof_pred = [x["pred"] for x in ordered_oof]

    oof_strict = entity_exact_breakdown(oof_gold, oof_pred)
    oof_token = token_typed_breakdown(oof_gold, oof_pred)
    risk_f1 = risk_weighted_f1(oof_strict)

    # Source-of-truth consistency: OOF gold counts MUST equal dataset counts.
    oof_gold_counts = {e:oof_strict["per_entity"][e]["gold"] for e in ENTITY_TYPES}
    if oof_gold_counts != dataset_audit["train_entity_counts"]:
        raise AssertionError(
            f"OOF gold counts differ from dataset counts: "
            f"{oof_gold_counts} vs {dataset_audit['train_entity_counts']}"
        )

    oof_file = os.path.join(model_dir, "oof_predictions.json")
    _atomic_json_dump(ordered_oof, oof_file)

    fold_assignment_file = os.path.join(model_dir, "fold_assignments.json")
    _atomic_json_dump([
        {
            "fold":r["fold"],
            "train_inner_record_ids":r["train_inner_record_ids"],
            "val_inner_record_ids":r["val_inner_record_ids"],
            "heldout_record_ids":r["heldout_record_ids"],
        }
        for r in fold_results
    ], fold_assignment_file, indent=2)

    # Persist complete CV before starting the final full-data training stage.
    cv_summary_file = os.path.join(model_dir, "cv_complete_summary.json")
    _atomic_json_dump({
        "model": cfg.name,
        "seed": seed,
        "oof_strict": oof_strict,
        "oof_token": oof_token,
        "risk_weighted_f1": float(risk_f1),
        "dataset_entity_counts": dataset_audit["train_entity_counts"],
    }, cv_summary_file, indent=2)
    cv_bundle = os.path.join(model_dir, f"{cfg.name.lower()}_cv_complete.zip")
    cv_files = [oof_file, fold_assignment_file, cv_summary_file]
    for fr in fold_results:
        cv_files.extend([fr.get("fold_summary_file"), fr.get("heldout_predictions_file")])
    _zip_named_files(cv_bundle, cv_files, model_dir)
    if _DRIVE_READY:
        for p in cv_files + [cv_bundle]:
            _backup_file_to_drive(p)
        print(f"[{cfg.name}] complete CV persisted before final blind-test training")
    else:
        _safe_download(cv_bundle, f"{cfg.name} complete CV backup")

    print("\nTraining full model and predicting BLIND test...")
    final = train_final_and_score_blind(
        cfg, tokenizer, train_docs, blind_docs, gold_xml,
        os.path.join(model_dir, "final_blind_test"), seed
    )

    payload = {
        "schema_version":"icacin_camera_ready_replication_v5",
        "seed":seed,
        "model_cfg":asdict(cfg),
        "label_mapping":{
            "raw_to_target":RAW_TO_TARGET,
            "excluded_raw_types":sorted(EXCLUDED_RAW_TYPES),
            "target_entities":ENTITY_TYPES,
        },
        "dataset_audit":dataset_audit,
        "tokenizer_window_audit":tokenizer_window_audit,
        "environment":collect_environment_info(),
        "source_files":{
            "train_xml":{"name":os.path.basename(train_xml),"sha256":_sha256_file(train_xml)},
            "blind_test_xml":{"name":os.path.basename(blind_xml),"sha256":_sha256_file(blind_xml)},
            "gold_test_xml":{"name":os.path.basename(gold_xml),"sha256":_sha256_file(gold_xml)},
        },
        "fold_policy":"Seeded document-level 5-fold CV; inner validation is drawn only from outer-training documents.",
        "checkpoint_policy":"Inner-validation token-typed micro-F1 only.",
        "metric_policy":"Strict entity and token-typed metrics are computed from the same one-shot held-out predictions.",
        "outer_prediction_policy":"Trainer validation compute_metrics is disabled during outer/blind inference; scoring is performed explicitly from merged predictions.",
        "cv_folds":fold_results,
        "fold_assignments_file":fold_assignment_file,
        "cv_complete_bundle":cv_bundle,
        "oof_predictions_file":oof_file,
        "oof_strict":oof_strict,
        "oof_token":oof_token,
        "token_entity_inflation":float(oof_token["overall"]["f1"] - oof_strict["overall"]["f1"]),
        "risk_weighted_f1":float(risk_f1),
        "final_blind_test":final,
    }

    result_file = os.path.join(model_dir, f"{cfg.name.lower()}_camera_ready_results.json")
    _atomic_json_dump(payload, result_file, indent=2)

    # Compact per-model archive: enough to reproduce all metrics/tables/figures
    # even if Colab dies before the next model starts.
    model_bundle = os.path.join(model_dir, f"{cfg.name.lower()}_complete_results.zip")
    model_files = [result_file, oof_file, fold_assignment_file, cv_summary_file, cv_bundle,
                   final.get("blind_prediction_file"), final.get("final_test_predictions_with_gold_file"),
                   final.get("final_training_summary_file")]
    for fr in fold_results:
        model_files.extend([fr.get("fold_summary_file"), fr.get("heldout_predictions_file")])
    _zip_named_files(model_bundle, model_files, model_dir)

    # Persist compact result artifacts + final model weights externally.
    if _DRIVE_READY:
        _backup_dir_to_drive(model_dir, include_large=False)
        if BACKUP_FINAL_MODEL_TO_DRIVE and os.path.isdir(final.get("final_model_dir", "")):
            print(f"[{cfg.name}] backing up final model weights to Google Drive...")
            _backup_dir_to_drive(final["final_model_dir"], include_large=True)
        print(f"[{cfg.name}] complete model results persisted to Google Drive")

    if DOWNLOAD_RESULTS_JSON:
        _safe_download(result_file, f"{cfg.name} summary JSON")
    if DOWNLOAD_EACH_MODEL_BUNDLE:
        _safe_download(model_bundle, f"{cfg.name} complete results bundle")

    return result_file

# ============================================================
# Cross-model table generator
# Every paper result table is derived from the SAME OOF predictions.
# ============================================================
def build_paper_tables(result_files, out_dir=None):
    rows = [json.load(open(p, encoding="utf-8")) for p in result_files]
    out_dir = out_dir or os.path.join(OUTPUT_ROOT, "paper_tables")
    os.makedirs(out_dir, exist_ok=True)

    def model_name(r): return r["model_cfg"]["name"]

    # Table 6
    t6=[]
    for r in rows:
        s=r["oof_strict"]["overall"]; t=r["oof_token"]["overall"]
        t6.append({
            "Model":model_name(r),
            "Strict_F1":s["f1"], "Strict_P":s["precision"], "Strict_R":s["recall"],
            "Token_F1":t["f1"], "Delta":t["f1"]-s["f1"],
            "RiskWeighted_F1":r["risk_weighted_f1"],
        })

    # Tables 7/8/9/10 in long form
    t7=[];t8=[];t9=[];t10=[]
    fold_metrics=[]
    final_overall=[]
    final_per_entity=[]
    for r in rows:
        m=model_name(r)
        for e in ENTITY_TYPES:
            s=r["oof_strict"]["per_entity"][e]
            t=r["oof_token"]["per_entity"][e]
            t7.append({"Model":m,"Entity":e,"Strict_F1":s["f1"],"P":s["precision"],"R":s["recall"]})
            t8.append({"Model":m,"Entity":e,"Token_F1":t["f1"],"P":t["precision"],"R":t["recall"]})
            t9.append({"Model":m,"Entity":e,"FP_share":s["fp_share"],"FN_share":s["fn_share"],
                       "TP":s["tp"],"FP":s["fp"],"FN":s["fn"]})
            t10.append({"Model":m,"Entity":e,"Gold":s["gold"],"Pred":s["pred"],
                        "Bias_pct":100.0*s["bias_pct"]})

        for frow in r["cv_folds"]:
            fold_metrics.append({
                "Model":m, "Fold":frow["fold"],
                "Strict_F1":frow["strict"]["overall"]["f1"],
                "Strict_P":frow["strict"]["overall"]["precision"],
                "Strict_R":frow["strict"]["overall"]["recall"],
                "Token_F1":frow["token"]["overall"]["f1"],
                "Best_inner_val_token_F1":frow.get("best_metric"),
                "Train_inner_docs":frow["train_inner_docs"],
                "Val_inner_docs":frow["val_inner_docs"],
                "Heldout_docs":frow["heldout_docs"],
                "Train_seconds":frow["train_seconds"],
            })

        fs=r["final_blind_test"]["strict"]
        ft=r["final_blind_test"]["token"]
        final_overall.append({
            "Model":m,
            "Strict_F1":fs["overall"]["f1"],
            "Strict_P":fs["overall"]["precision"],
            "Strict_R":fs["overall"]["recall"],
            "Token_F1":ft["overall"]["f1"],
            "Token_P":ft["overall"]["precision"],
            "Token_R":ft["overall"]["recall"],
        })
        for e in ENTITY_TYPES:
            se=fs["per_entity"][e]; te=ft["per_entity"][e]
            final_per_entity.append({
                "Model":m,"Entity":e,
                "Strict_F1":se["f1"],"Strict_P":se["precision"],"Strict_R":se["recall"],
                "Token_F1":te["f1"],"Token_P":te["precision"],"Token_R":te["recall"],
                "TP":se["tp"],"FP":se["fp"],"FN":se["fn"],
                "Gold":se["gold"],"Pred":se["pred"],"Bias_pct":100.0*se["bias_pct"],
            })

    def write_csv(name, data):
        path=os.path.join(out_dir,name)
        with open(path,"w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f,fieldnames=list(data[0].keys()))
            w.writeheader();w.writerows(data)
        return path

    paths_out={
        "table6":write_csv("table6_overall.csv",t6),
        "table7":write_csv("table7_strict_per_entity.csv",t7),
        "table8":write_csv("table8_token_per_entity.csv",t8),
        "table9":write_csv("table9_errors.csv",t9),
        "table10":write_csv("table10_bias.csv",t10),
        "fold_metrics":write_csv("fold_metrics.csv",fold_metrics),
        "final_test_overall":write_csv("final_test_overall.csv",final_overall),
        "final_test_per_entity":write_csv("final_test_per_entity.csv",final_per_entity),
    }

    # Consistency checks across models.
    gold_by_model={
        model_name(r):{e:r["oof_strict"]["per_entity"][e]["gold"] for e in ENTITY_TYPES}
        for r in rows
    }
    unique_gold={tuple(x[e] for e in ENTITY_TYPES) for x in gold_by_model.values()}
    if len(unique_gold) != 1:
        raise AssertionError(f"Gold entity counts differ across models: {gold_by_model}")

    checks={
        "all_models_same_gold_counts":True,
        "gold_counts":gold_by_model,
        "table9_and_table7_same_source":True,
        "tables_6_to_10_source":"per-model unified OOF predictions",
    }
    check_path=os.path.join(out_dir,"consistency_checks.json")
    _atomic_json_dump(checks, check_path, indent=2)
    paths_out["checks"]=check_path

    print("Paper tables written to:", out_dir)
    return paths_out

# ============================================================
# Compact archival bundle
# Preserves everything needed to regenerate metrics/tables/figures
# without retraining. Model weights are intentionally kept separate
# because they are much larger.
# ============================================================
def build_results_bundle(result_files, table_files, out_dir=None):
    out_dir = out_dir or OUTPUT_ROOT
    os.makedirs(out_dir, exist_ok=True)
    bundle_path = os.path.join(out_dir, "icacin_results_bundle.zip")
    manifest_path = os.path.join(out_dir, "run_manifest.json")

    artifacts = []
    def add_path(path, role):
        if path and os.path.exists(path):
            artifacts.append({
                "path":path,
                "role":role,
                "size_bytes":os.path.getsize(path),
                "sha256":_sha256_file(path),
            })

    loaded = []
    for rf in result_files:
        r = json.load(open(rf, encoding="utf-8"))
        loaded.append(r)
        add_path(rf, "model_summary_json")
        add_path(r.get("oof_predictions_file"), "oof_predictions")
        add_path(r.get("fold_assignments_file"), "fold_assignments")
        final = r.get("final_blind_test", {})
        add_path(final.get("blind_prediction_file"), "blind_predictions_frozen_before_gold")
        add_path(final.get("final_test_predictions_with_gold_file"), "final_test_scored_predictions")
        add_path(final.get("final_training_summary_file"), "final_training_history")
        for fold in r.get("cv_folds", []):
            add_path(fold.get("fold_summary_file"), f"fold_{fold.get('fold')}_summary")
            add_path(fold.get("heldout_predictions_file"), f"fold_{fold.get('fold')}_heldout_predictions")

    for name,path in table_files.items():
        add_path(path, f"paper_output_{name}")

    manifest = {
        "schema_version":"icacin_archival_bundle_v2",
        "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":collect_environment_info(),
        "models":[r["model_cfg"] for r in loaded],
        "source_hashes":loaded[0].get("source_files") if loaded else None,
        "artifacts":[{k:v for k,v in a.items() if k != "path"} | {"archive_name":os.path.relpath(a["path"], OUTPUT_ROOT)} for a in artifacts],
        "note":"This bundle excludes final model weights. It contains OOF predictions, blind-test predictions, gold-scored final-test predictions, exact fold membership, training logs, metrics, tables, checks, and run metadata.",
    }
    _atomic_json_dump(manifest, manifest_path, indent=2)
    add_path(manifest_path, "run_manifest")

    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        seen=set()
        for a in artifacts:
            p=a["path"]
            if p in seen or not os.path.exists(p):
                continue
            seen.add(p)
            arc=os.path.relpath(p, OUTPUT_ROOT)
            z.write(p, arcname=arc)

    print("Archival results bundle:", bundle_path)
    if DOWNLOAD_RESULTS_BUNDLE and files is not None:
        _safe_download(bundle_path, "all-model archival bundle")
    return bundle_path

# ============================================================
# Model configurations retained from older implementation
# ============================================================
clinical_cfg = ModelCfg(
    name="ClinicalBERT",
    model_id="emilyalsentzer/Bio_ClinicalBERT",
    batch_size=4, grad_accum=2, lr=3e-5,
    window_boosts={"LOCATION":1.3,"PHONE":1.0},
    use_class_weights=True,
    class_weight_multipliers={"LOCATION":1.5,"PHONE":1.1},
)

biobert_cfg = ModelCfg(
    name="BioBERT",
    model_id="dmis-lab/biobert-base-cased-v1.1",
    batch_size=4, grad_accum=2, lr=3e-5,
    window_boosts={"LOCATION":1.3,"PHONE":1.0},
    use_class_weights=True,
    class_weight_multipliers={"LOCATION":1.5,"PHONE":1.1},
)

roberta_cfg = ModelCfg(
    name="RoBERTaLarge",
    model_id="roberta-large",
    batch_size=2, grad_accum=4, lr=2e-5,
    window_boosts={"LOCATION":1.3,"PHONE":1.0},
    use_class_weights=True,
    class_weight_multipliers={"LOCATION":1.5,"PHONE":1.1},
)

print("Setup complete (V5).")
print("Target entities:", ENTITY_TYPES)
print("Run prepare_corpora() FIRST and inspect the audit before starting GPU training.")


## 1. Data audit — run this before any GPU training
Expected source split: 669 annotated training records; 220 blind test records; gold test is not opened here. The first model run will also prompt to mount Google Drive for crash-safe persistence.


In [ ]:
train_docs, blind_docs, audit = prepare_corpora()
print('\nSTOP if these counts look wrong before training.')

## 2. Run RoBERTa-Large first
Use this as the replication diagnostic. Do not change settings after seeing held-out or blind-test results.

In [ ]:
roberta_result = run_model(roberta_cfg, seed=SEED)
print('RoBERTa result:', roberta_result)

## 3. If RoBERTa is coherent, run the two BERT models with the same frozen protocol

In [ ]:
clinical_result = run_model(clinical_cfg, seed=SEED)
print('ClinicalBERT result:', clinical_result)

In [ ]:
# ============================================================
# BioBERT legacy config compatibility fix
# ============================================================

from transformers import AutoConfig, BertConfig

BIOBERT_MODEL_ID = "dmis-lab/biobert-base-cased-v1.1"

# Save the normal AutoConfig loader only once, so rerunning this
# cell does not create recursive patches.
if not hasattr(AutoConfig, "_original_from_pretrained_biobert_fix"):
    AutoConfig._original_from_pretrained_biobert_fix = AutoConfig.from_pretrained


def _patched_autoconfig_from_pretrained(model_id, *args, **kwargs):
    if model_id == BIOBERT_MODEL_ID:
        print(
            "BioBERT legacy config detected -> "
            "loading explicitly as BertConfig"
        )

        # Read BioBERT's own legacy config, but tell Transformers
        # explicitly that the architecture is BERT.
        config = BertConfig.from_pretrained(
            model_id,
            *args,
            **kwargs
        )

        print("Config class:", config.__class__.__name__)
        print("Model type:", config.model_type)
        print("Hidden size:", config.hidden_size)
        print("Layers:", config.num_hidden_layers)

        return config

    return AutoConfig._original_from_pretrained_biobert_fix(
        model_id,
        *args,
        **kwargs
    )


AutoConfig.from_pretrained = staticmethod(
    _patched_autoconfig_from_pretrained
)

print("BioBERT legacy model-config fix loaded.")

In [ ]:
biobert_result = run_model(biobert_cfg, seed=SEED)
print("BioBERT result:", biobert_result)

## 4. Generate all paper tables + final all-model archive
Each completed model has already downloaded its own compact results ZIP and been persisted to Google Drive. This final step combines all three into paper tables and one cross-model archive.


In [ ]:
paper_table_files = build_paper_tables(
    [roberta_result, clinical_result, biobert_result]
)
print(json.dumps(paper_table_files, indent=2))

results_bundle = build_results_bundle(
    [roberta_result, clinical_result, biobert_result],
    paper_table_files
)
print('SAVE THIS ZIP:', results_bundle)
